# Notebook De Jean-Baptiste CHEZE

### Ce qu'il faut savoir sur le DataSet : 
- Les données viennent du site EaseMyTrip (données secondaires).
- Elles ont été récoltées via Octoparse, un outil de web scraping.
- Période de collecte : 50 jours, du 11 février au 31 mars 2022.

Données séparées selon la classe du billet :
- Économie
- Business

Nombre total d’observations : 300 261 options de vol.

### Problématique / Questions de recherche

1. Le prix varie-t-il selon la compagnie aérienne ?
- Utiliser un test ANOVA pour comparer les prix entre compagnies.

2. Acheter 1 ou 2 jours avant le départ augmente-t-il le prix ?
- Étudier la relation entre days_left et le prix (analyse statistique ou graphique).

3. Le prix dépend-il de l’heure de départ et de l’heure d’arrivée ?
- Catégoriser les heures (matin, après-midi, nuit…) puis comparer les prix via tests ou modèle linéaire.

4. Le prix change-t-il selon la ville de départ et la ville d’arrivée ?
- Comparer les prix selon les routes (ex. Delhi–Mumbai, Chennai–Bangalore…).

5. Le prix est-il plus élevé en business qu’en economy, et de combien ?
- Comparer les deux classes à l’aide d’un test statistique (moyennes ou distributions).

| N° | Variable          | Type               | Signification |
|----|-------------------|--------------------|----------------|
| 1  | Airline           | Catégorielle       | Nom de la compagnie aérienne (6 compagnies différentes). |
| 2  | Flight            | Catégorielle       | Code du vol (identifiant du vol). |
| 3  | Source City       | Catégorielle       | Ville de départ du vol (6 villes uniques). |
| 4  | Departure Time    | Catégorielle dérivée | Créée en regroupant les heures de départ en plages horaires (6 labels). |
| 5  | Stops             | Catégorielle       | Nombre d’escales entre la ville de départ et d’arrivée (3 valeurs possibles). |
| 6  | Arrival Time      | Catégorielle dérivée | Créée en regroupant les heures d’arrivée en plages horaires (6 labels). |
| 7  | Destination City  | Catégorielle       | Ville d’arrivée du vol (6 villes uniques). |
| 8  | Class             | Catégorielle       | Classe du billet : Economy ou Business. |
| 9  | Duration          | Continue           | Durée totale du vol en heures. |
| 10 | Days Left         | Continue dérivée   | Nombre de jours restants entre la date de réservation et la date du vol. |
| 11 | Price             | Continue (variable cible) | Prix du billet d’avion. |


In [23]:
import pandas as pd

## Analyse Exploratoire (EDA)

In [2]:
df = pd.read_csv(filepath_or_buffer="../data/Clean_Dataset.csv",
                 sep = ",")

In [4]:
df.head(3)

,Unnamed: 0,airline,flight,source_city,departure_time,stops,arrival_time,destination_city,class,duration,days_left,price
0,0,SpiceJet,SG-8709,Delhi,Evening,zero,Night,Mumbai,Economy,2.17,1,5953
1,1,SpiceJet,SG-8157,Delhi,Early_Morning,zero,Morning,Mumbai,Economy,2.33,1,5953
2,2,AirAsia,I5-764,Delhi,Early_Morning,zero,Early_Morning,Mumbai,Economy,2.17,1,5956


In [6]:
df.drop("Unnamed: 0",
        axis=1,
        inplace = True)

In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 300153 entries, 0 to 300152
Data columns (total 11 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   airline           300153 non-null  object 
 1   flight            300153 non-null  object 
 2   source_city       300153 non-null  object 
 3   departure_time    300153 non-null  object 
 4   stops             300153 non-null  object 
 5   arrival_time      300153 non-null  object 
 6   destination_city  300153 non-null  object 
 7   class             300153 non-null  object 
 8   duration          300153 non-null  float64
 9   days_left         300153 non-null  int64  
 10  price             300153 non-null  int64  
dtypes: float64(1), int64(2), object(8)
memory usage: 25.2+ MB


In [20]:
df.describe()

,duration,days_left,price
count,300153.000000,300153.000000,300153.000000
mean,12.221021,26.004751,20889.660523
std,7.191997,13.561004,22697.767366
min,0.830000,1.000000,1105.000000
25%,6.830000,15.000000,4783.000000
50%,11.250000,26.000000,7425.000000
75%,16.170000,38.000000,42521.000000
max,49.830000,49.000000,123071.000000


In [19]:
for value in df.select_dtypes(include=['object']).columns : 
    print(f"======{value}======")
    print(df[f"{value}"].value_counts())

======airline======
airline
Vistara      127859
Air_India     80892
Indigo        43120
GO_FIRST      23173
AirAsia       16098
SpiceJet       9011
Name: count, dtype: int64
======flight======
flight
UK-706     3235
UK-772     2741
UK-720     2650
UK-836     2542
UK-822     2468
           ... 
6E-2914       1
G8-107        1
SG-9974       1
6E-865        1
SG-8339       1
Name: count, Length: 1561, dtype: int64
======source_city======
source_city
Delhi        61343
Mumbai       60896
Bangalore    52061
Kolkata      46347
Hyderabad    40806
Chennai      38700
Name: count, dtype: int64
======departure_time======
departure_time
Morning          71146
Early_Morning    66790
Evening          65102
Night            48015
Afternoon        47794
Late_Night        1306
Name: count, dtype: int64
======stops======
stops
one            250863
zero            36004
two_or_more     13286
Name: count, dtype: int64
======arrival_time======
arrival_time
Night            91538
Evening          78323
Mo

# Preprocessing

**On va faire du binaire pour les variables catégorielles a 2 modalités**

In [21]:
df["class"] = df["class"].replace({"Business" : 1,
                                   "Economy" : 0})

C:\Users\jbche\AppData\Local\Temp\ipykernel_27180\4171413487.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df["class"] = df["class"].replace({"Business" : 1,


**On va faire du Ordinal encoding pour les variables caétgorielles qui ont une hierachie**

In [24]:
df["stops"] = df["stops"].replace({
    "zero": 0,
    "one": 1,
    "two_or_more": 2
})

C:\Users\jbche\AppData\Local\Temp\ipykernel_27180\2358550755.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df["stops"] = df["stops"].replace({


**On va faire du One Hot encoding pour les variables caétgorielles sasn hiearchie**

In [ ]:
def one_hot_encoding(colonnes: list[str], dataframe: pd.DataFrame) -> pd.DataFrame:
    df = dataframe.copy()
    for col in colonnes:
        dummies = pd.get_dummies(df[col], prefix=col)
        df = df.join(dummies)
    return df

one_hot_encoding([airline,source_city,arrival_time,destination_city,departure_time])
